## 1. Importação de Bibliotecas

In [ ]:
import warnings
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, Markdown

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    silhouette_score,
)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

## 2. Configuração do Ambiente

In [ ]:
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

BASE_DIR = Path(".")
FIGURES_DIR = BASE_DIR / "figures"
TABLES_DIR = BASE_DIR / "tables"

FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)

## 3. Funções Auxiliares

In [ ]:
def load_excel(filename: str) -> pd.DataFrame:
    path = BASE_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    return pd.read_excel(path)


def save_figure(fig: plt.Figure, filename: str) -> None:
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")

## 4. Carregamento dos Dados

In [ ]:
df_ativos = load_excel("Estudantes_ativos_EP_2025.xlsx")
df_inativos = load_excel("Estudantes_inativos_EP_2025.xlsx")
df_concat = load_excel("Estudantes_EP_2025_concat.xlsx")

print("Shape df_ativos:", df_ativos.shape)
print("Shape df_inativos:", df_inativos.shape)
print("Shape df_concat:", df_concat.shape)

## 5. Filtro por Ano de Ingresso

In [ ]:
df_inativos = df_inativos[df_inativos['Ano_Ingresso'] >= 2012]
df_concat = df_concat[df_concat['Ano_Ingresso'] >= 2012]

print("\nApós filtro (Ano_Ingresso >= 2012):")
print("Shape df_inativos:", df_inativos.shape)
print("Shape df_concat:", df_concat.shape)

## 6. Resumo de IRA

In [ ]:
def resumo_ira(df: pd.DataFrame, nome_base: str) -> dict:
    total = len(df)
    if "IRA" not in df.columns:
        return {
            "Base": nome_base,
            "Registros_totais": int(total),
            "Registros_IRA_maior_0": 0,
            "Removidos_IRA_0_ou_na": int(total),
        }
    ira_col = pd.to_numeric(df["IRA"], errors="coerce")
    apos_filtro = (ira_col > 0).sum()
    removidos = total - apos_filtro
    return {
        "Base": nome_base,
        "Registros_totais": int(total),
        "Registros_IRA_maior_0": int(apos_filtro),
        "Removidos_IRA_0_ou_na": int(removidos),
    }

tabela_1_list = [
    resumo_ira(df_ativos, "Ativos"),
    resumo_ira(df_inativos, "Inativos"),
    resumo_ira(df_concat, "Concat"),
]

tabela_1 = pd.DataFrame(tabela_1_list)
display(tabela_1)

tabela_1_path = os.path.join("tables", "tabela_1_resumo_ira.xlsx")
tabela_1.to_excel(tabela_1_path, index=False)

## 7. Distribuição por Estado

In [ ]:
if "Estado" in df_concat.columns:
    dist_estado = (
        df_concat["Estado"]
        .astype(str)
        .str.strip()
        .replace({"nan": np.nan})
        .dropna()
        .value_counts()
        .sort_values(ascending=True)
        .reset_index()
    )
    dist_estado.columns = ["Estado", "Quantidade"]

    fig, ax = plt.subplots(figsize=(8, max(4, 0.3 * len(dist_estado))))
    sns.barplot(data=dist_estado, x="Quantidade", y="Estado", ax=ax)
    ax.set_xlabel("Número de estudantes")
    ax.set_ylabel("UF")

    save_figure(fig, "fig_1_perfil_estado.png")
    plt.show()

## 8. Distribuição por Sexo ao Longo dos Anos

In [ ]:
if {"Sexo", "Ano_Ingresso"}.issubset(df_concat.columns):
    tmp = (
        df_concat.dropna(subset=["Sexo", "Ano_Ingresso"])
        .groupby(["Ano_Ingresso", "Sexo"])
        .size()
        .reset_index(name="Quantidade")
    )
    total_ano = tmp.groupby("Ano_Ingresso")["Quantidade"].transform("sum")
    tmp["Proporcao"] = tmp["Quantidade"] / total_ano * 100

    fig, ax = plt.subplots(figsize=(8, 4))
    for sexo, grupo in tmp.groupby("Sexo"):
        ax.plot(grupo["Ano_Ingresso"], grupo["Proporcao"], marker="o", label=sexo)
    ax.set_xlabel("Ano de ingresso")
    ax.set_ylabel("Proporção de estudantes (%)")
    ax.legend(title="Sexo")

    save_figure(fig, "fig_02_sexo_por_ano_ingresso.png")
    plt.show()

## 9. Ingressos vs Desligamentos por Ano

In [ ]:
ingressantes_ano = (
    df_concat.dropna(subset=["Ano_Ingresso"])
    .groupby("Ano_Ingresso")
    .size()
    .reset_index(name="Ingressantes")
)

if "Ano_Egresso" in df_inativos.columns:
    desligados_ano = (
        df_inativos.dropna(subset=["Ano_Egresso"])
        .groupby("Ano_Egresso")
        .size()
        .reset_index(name="Desligados")
    )
else:
    desligados_ano = pd.DataFrame(columns=["Ano_Egresso", "Desligados"])

df_fluxo = pd.merge(
    ingressantes_ano,
    desligados_ano,
    left_on="Ano_Ingresso",
    right_on="Ano_Egresso",
    how="left",
)
df_fluxo["Desligados"] = df_fluxo["Desligados"].fillna(0)

fig, ax = plt.subplots(figsize=(9, 4))
largura = 0.4
anos = df_fluxo["Ano_Ingresso"].astype(int).values
idx = np.arange(len(anos))

ax.bar(idx - largura / 2, df_fluxo["Ingressantes"], width=largura, label="Ingressantes")
ax.bar(idx + largura / 2, df_fluxo["Desligados"], width=largura, label="Desligados/egressos")

ax.set_xticks(idx)
ax.set_xticklabels(anos, rotation=45)
ax.set_xlabel("Ano")
ax.set_ylabel("Número de estudantes")
ax.legend()

save_figure(fig, "fig_3_ingressos_desligamentos_por_ano.png")
plt.show()

## 10. Tipo de Ingresso por Ano (Ativos)

In [ ]:
if {"Tipo_Ingresso", "Ano_Ingresso"}.issubset(df_ativos.columns):
    dist_tipo = (
        df_ativos.dropna(subset=["Tipo_Ingresso", "Ano_Ingresso"])
        .groupby(["Ano_Ingresso", "Tipo_Ingresso"])
        .size()
        .reset_index(name="Quantidade")
    )

    tipos = dist_tipo["Tipo_Ingresso"].astype(str).unique()
    anos = sorted(dist_tipo["Ano_Ingresso"].astype(int).unique())

    fig, ax = plt.subplots(figsize=(9, 5))
    bottom = np.zeros(len(anos))

    for tipo in tipos:
        subset = dist_tipo[dist_tipo["Tipo_Ingresso"] == tipo]
        valores = [subset[subset["Ano_Ingresso"] == ano]["Quantidade"].sum() for ano in anos]
        ax.bar(anos, valores, bottom=bottom, label=tipo)
        bottom += valores

    ax.set_xlabel("Ano de ingresso")
    ax.set_ylabel("Número de estudantes")
    ax.set_xticks(anos)
    ax.set_xticklabels(anos)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles,
        labels,
        title="Tipo de ingresso",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.25),
        ncol=3,
        frameon=False,
    )

    plt.tight_layout(rect=[0, 0.18, 1, 1])

    save_figure(fig, "fig_4_tipo_ingresso_por_ano.png")
    plt.show()

## 11. Indicadores Socioeconômicos

In [ ]:
indicadores = [
    "Pessoa_PPI", 
    "Renda_Superior_1_5", 
    "EM_Escola_Publica", 
    "PCD"
]
present_cols = [c for c in indicadores if c in df_ativos.columns]

if present_cols:
    df_ind = df_ativos.copy()
    df_ind = df_ind[pd.to_numeric(df_ind["IRA"], errors="coerce") > 0]

    for col in present_cols:
        df_ind[col] = pd.to_numeric(df_ind[col], errors="coerce")

    prop_list = []
    for col in present_cols:
        vc = df_ind[col].value_counts(normalize=True) * 100
        for cat, val in vc.items():
            prop_list.append({
                "Indicador": col,
                "Categoria": str(cat),
                "Proporcao": val
            })

    df_prop = pd.DataFrame(prop_list)

    fig, ax = plt.subplots(figsize=(14, 4))
    sns.barplot(data=df_prop, x="Indicador", y="Proporcao", hue="Categoria", ax=ax)
    ax.set_ylabel("Proporção (%)")
    ax.set_xlabel("Indicador")
    plt.xticks(rotation=30)

    save_figure(fig, "fig_5_indicadores_socioeconomicos.png")
    plt.show()

## 12. IRA por Indicadores Socioeconômicos

In [ ]:
if present_cols and "IRA" in df_ind.columns:
    for col in present_cols:
        fig, ax = plt.subplots(figsize=(6, 4))
        df_ind_clean = df_ind.dropna(subset=[col, "IRA"])
        df_ind_clean[col] = df_ind_clean[col].astype(str)
        
        sns.boxplot(data=df_ind_clean, x=col, y="IRA", ax=ax)
        ax.set_xlabel(col)
        ax.set_ylabel("IRA")
        
        save_figure(fig, f"fig_6_ira_por_{col}.png")
        plt.show()

## 13. Vulnerabilidade por Perfil

In [ ]:
if "Perfil" in df_ativos.columns and present_cols:
    df_vuln = df_ativos.copy()

    for col in present_cols:
        df_vuln[col] = pd.to_numeric(df_vuln[col], errors="coerce")

    cond_list = []

    if "Pessoa_PPI" in present_cols:
        cond_list.append(df_vuln["Pessoa_PPI"] == 1)

    if "Renda_Superior_1_5" in present_cols:
        cond_list.append(df_vuln["Renda_Superior_1_5"] == 0)

    if "EM_Escola_Publica" in present_cols:
        cond_list.append(df_vuln["EM_Escola_Publica"] == 1)

    if "PCD" in present_cols:
        cond_list.append(df_vuln["PCD"] == 1)

    if cond_list:
        df_vuln["Vulneravel"] = 0
        for cond in cond_list:
            df_vuln.loc[cond, "Vulneravel"] = 1
        
        vuln_perfil = (
            df_vuln.groupby("Perfil")["Vulneravel"]
            .agg(["sum", "count"])
            .reset_index()
        )
        vuln_perfil["Proporcao"] = (vuln_perfil["sum"] / vuln_perfil["count"]) * 100
        
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.barplot(data=vuln_perfil, x="Perfil", y="Proporcao", ax=ax)
        ax.set_ylabel("Proporção com vulnerabilidade (%)")
        ax.set_xlabel("Perfil")
        
        save_figure(fig, "fig_7_vulnerabilidade_por_perfil.png")
        plt.show()